In [1]:
import pandas as pd 
import numpy as np

In [36]:
df = pd.read_csv(r"D:\All_Final_Ml_Models_Model - V2\Battery_temperature_predication_end_to_end\data\raw\train.csv")

In [37]:
features = [
        "timestamp", "SOH", "Cycle_Count", "SOC",
        "Battery_1_Volt", "Battery_2_Volt", "Battery_3_Volt", "Battery_4_Volt",
        "Battery_5_Volt", "Battery_6_Volt", "Battery_7_Volt", "Battery_8_Volt",
        "Battery_9_Volt", "Battery_10_Volt", "Battery_11_Volt", "Battery_12_Volt",
        "Battery_13_Volt", "Battery_14_Volt", "Battery_15_Volt", "Battery_16_Volt",
        "temperature_1", "temperature_2", "temperature_3", "temperature_4",
        "temperature_5", "temperature_6",
        "Battery_current", "Residual_battery_energy", "Remaining_Capacity",
        "Full_Capacity", "Full_battery_energy", "error_code", "Battery_pack_total_voltage"
    ]

df = df[features]

df = df.dropna(subset=["timestamp"])

df["timestamp"] = pd.to_datetime(df["timestamp"],errors="coerce",utc=True)

# 2️ Sort by time
df = df.sort_values("timestamp").reset_index(drop=True)

# Compute time difference (in seconds)
df["time_diff_sec"] = df["timestamp"].diff().dt.total_seconds()

# Keep only rows with time gap in [5s, 15s]
df = df[
    (df["time_diff_sec"].between(5, 15)) |
    (df["time_diff_sec"].isna())   # keep first row
]

#imputing the missing values 

# Temperature
temp_cols = [f"temperature_{i}" for i in range(1, 7)]
df[temp_cols] = df[temp_cols].apply(
    lambda row: row.fillna(row.mean()), axis=1
)

# Voltages
volt_cols = [c for c in df.columns if c.startswith("Battery_") and c.endswith("_Volt")]
df[volt_cols] = df[volt_cols].apply(
    lambda row: row.fillna(row.mean()), axis=1
)

#conversion in into mV to V
df[volt_cols] = df[volt_cols] / 1000.0

# Pack voltage
df["Battery_pack_total_voltage"] = df[volt_cols].sum(axis=1)

# Stateful signals
df[["SOC", "SOH", "Cycle_Count"]] = df[["SOC", "SOH", "Cycle_Count"]].ffill()

# Capacity
df[["Remaining_Capacity", "Full_Capacity"]] = df[["Remaining_Capacity", "Full_Capacity"]].ffill()
df.dropna(inplace=True)

df.reset_index(drop=True, inplace=True)

In [38]:
df.shape

(10815, 34)

In [28]:
features = [
        "timestamp", "SOH", "Cycle_Count", "SOC",
        "Battery_1_Volt", "Battery_2_Volt", "Battery_3_Volt", "Battery_4_Volt",
        "Battery_5_Volt", "Battery_6_Volt", "Battery_7_Volt", "Battery_8_Volt",
        "Battery_9_Volt", "Battery_10_Volt", "Battery_11_Volt", "Battery_12_Volt",
        "Battery_13_Volt", "Battery_14_Volt", "Battery_15_Volt", "Battery_16_Volt",
        "temperature_1", "temperature_2", "temperature_3", "temperature_4",
        "temperature_5", "temperature_6", "Maximum_temperature", "Minimum_temperature",
        "Battery_current", "Residual_battery_energy", "Remaining_Capacity",
        "Full_Capacity", "Full_battery_energy", "Battery_pack_total_voltage"
    ]
df = df[features]

In [29]:
df.head()

,timestamp,SOH,Cycle_Count,SOC,Battery_1_Volt,Battery_2_Volt,Battery_3_Volt,Battery_4_Volt,Battery_5_Volt,Battery_6_Volt,...,temperature_5,temperature_6,Maximum_temperature,Minimum_temperature,Battery_current,Residual_battery_energy,Remaining_Capacity,Full_Capacity,Full_battery_energy,Battery_pack_total_voltage
0,2026-02-04T15:34:17.987+00:00,100.0,4.0,100.0,3334.0,3334.0,3334.0,3334.0,3335.0,3335.0,...,28.0,28.0,NaN,NaN,0.0,3199.0,59.0,54.0,2881.0,53.3
1,2026-02-04T09:50:25.781+00:00,100.0,4.0,100.0,3342.0,3342.0,3342.0,3342.0,3342.0,3342.0,...,29.0,29.0,NaN,NaN,0.0,3202.0,59.0,54.0,2888.0,53.4
2,2026-02-04T03:01:08.699+00:00,100.0,2.0,96.0,3323.0,3323.0,3322.0,3322.0,3323.0,3323.0,...,22.0,22.0,NaN,NaN,0.0,1609.0,30.0,40.0,2153.0,53.1
3,2026-02-03T18:45:16.659+00:00,100.0,2.0,96.0,3326.0,3325.0,3325.0,3325.0,3326.0,3326.0,...,26.0,26.0,NaN,NaN,0.0,1630.0,30.0,40.0,2155.0,53.2
4,2026-02-04T00:33:10.969+00:00,100.0,2.0,96.0,3323.0,3323.0,3323.0,3323.0,3324.0,3324.0,...,23.0,23.0,NaN,NaN,0.0,1615.0,30.0,40.0,2154.0,53.1


In [39]:
df.isnull().sum()

timestamp                     0
SOH                           0
Cycle_Count                   0
SOC                           0
Battery_1_Volt                0
Battery_2_Volt                0
Battery_3_Volt                0
Battery_4_Volt                0
Battery_5_Volt                0
Battery_6_Volt                0
Battery_7_Volt                0
Battery_8_Volt                0
Battery_9_Volt                0
Battery_10_Volt               0
Battery_11_Volt               0
Battery_12_Volt               0
Battery_13_Volt               0
Battery_14_Volt               0
Battery_15_Volt               0
Battery_16_Volt               0
temperature_1                 0
temperature_2                 0
temperature_3                 0
temperature_4                 0
temperature_5                 0
temperature_6                 0
Battery_current               0
Residual_battery_energy       0
Remaining_Capacity            0
Full_Capacity                 0
Full_battery_energy           0
error_co

In [18]:
df.dropna(inplace=True)

In [20]:
df.shape

(0, 35)